# 2.1 — Choisir et explorer les données

1. Charger les données et afficher cinq exemples (texte, catégories, positions).
2. Compter les documents et les mentions par catégorie dans chaque partition.
3. Vérifier dix mentions de TRAIN : leurs positions doivent retrouver exactement les mots annotés.
4. Présenter la fiche du corpus.

**Corpus** : BC5CDR (`CDR.Corpus.v010516`, NCBI). **Catégories** : `Chemical` et `Disease`. Lecture des fichiers : `src/cdr.py`.

In [1]:
import sys, json, random
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.cdr import CATEGORIES, FICHIERS, charger_cdr, decouper_phrases, sauvegarder_jsonl

DOSSIER_CDR = RACINE / "CDR_Data" / "CDR.Corpus.v010516"
DATA = RACINE / "data"
RESULTATS = RACINE / "resultats" / "extraction"
SEED = 42

CORPUS_NOM = "BC5CDR"
print("Catégories :", CATEGORIES)
print("Fichiers   :", FICHIERS)

Catégories : ['Chemical', 'Disease']
Fichiers   : {'train': 'CDR_TrainingSet.PubTator.txt', 'validation': 'CDR_DevelopmentSet.PubTator.txt', 'test': 'CDR_TestSet.PubTator.txt'}


## Étape 1 — Charger les données et afficher cinq exemples

Le texte d'un document est `titre + " " + résumé` : les positions des mentions portent sur ce texte. Toutes les positions sont vérifiées au chargement.

In [2]:
corpus = charger_cdr(DOSSIER_CDR)
for nom, docs in corpus.items():
    print(f"{nom:<11} {len(docs)} documents chargés, positions vérifiées")

train       500 documents chargés, positions vérifiées
validation  500 documents chargés, positions vérifiées
test        500 documents chargés, positions vérifiées


Chaque mention est affichée sous la forme `[mention]<Type>`, puis listée avec ses positions.

In [3]:
def surligner(texte, mentions):
    morceaux, curseur = [], 0
    for m in mentions:
        morceaux.append(texte[curseur:m.debut])
        morceaux.append(f"[{texte[m.debut:m.fin]}]<{m.type[:4]}>")
        curseur = m.fin
    morceaux.append(texte[curseur:])
    return "".join(morceaux)

rng = random.Random(SEED)
exemples = rng.sample(corpus["train"], 5)
for d in exemples:
    print(f"=== PMID {d.document_id} ({d.partition}) — {len(d.mentions)} mentions")
    print(surligner(d.texte, d.mentions)[:700], "...")
    print()

=== PMID 19889778 (train) — 19 mentions
Differential impact of immune escape mutations G145R and P120T on the replication of [lamivudine]<Chem>-resistant [hepatitis B virus e antigen]<Chem>-positive and -negative strains. Immune escape variants of the [hepatitis B]<Dise> virus (HBV) represent an emerging clinical challenge, because they can be associated with vaccine escape, HBV reactivation, and failure of diagnostic tests. Recent data suggest a preferential selection of immune escape mutants in distinct peripheral blood leukocyte compartments of infected individuals. We therefore systematically analyzed the functional impact of the most prevalent immune escape variants, the sG145R and sP120T mutants, on the viral replication eff ...

=== PMID 11581460 (train) — 12 mentions
Bladder [retention of urine]<Dise> as a result of continuous intravenous infusion of [fentanyl]<Chem>: 2 case reports. Sedation has been commonly used in the neonate to decrease the stress and [pain]<Dise> from the

In [4]:
pd.DataFrame([
    {"PMID": d.document_id, "debut": m.debut, "fin": m.fin, "surface": m.texte,
     "categorie": m.type, "MeSH": m.identifiants, "composite": m.composite}
    for d in exemples[:2] for m in d.mentions
])

,PMID,debut,fin,surface,categorie,MeSH,composite
0,19889778,85,95,lamivudine,Chemical,D019259,False
1,19889778,106,133,hepatitis B virus e antigen,Chemical,D006513,False
2,19889778,196,207,hepatitis B,Disease,D006509,False
3,19889778,774,784,lamivudine,Chemical,D019259,False
4,19889778,786,789,LAM,Chemical,D019259,False
5,19889778,798,803,HBeAg,Chemical,D006513,False
6,19889778,876,879,LAM,Chemical,D019259,False
7,19889778,941,946,HBeAg,Chemical,D006513,False
8,19889778,963,968,HBeAg,Chemical,D006513,False
9,19889778,1083,1088,HBsAg,Chemical,D006514,False


## Étape 2 — Compter documents et mentions par partition

On compte aussi :
- les **mentions composites** (« ovarian and testicular cancer ») : deux maladies annotées comme une seule mention ;
- les **phrases sans mention** : utiles pour repérer les fausses détections.

In [5]:
lignes = []
for nom, docs in corpus.items():
    mentions = [m for d in docs for m in d.mentions]
    par_type = Counter(m.type for m in mentions)
    n_phrases = sans_mention = 0
    for d in docs:
        phrases = decouper_phrases(d.texte)
        n_phrases += len(phrases)
        sans_mention += sum(1 for a, b in phrases if not any(a <= m.debut < b for m in d.mentions))
    lignes.append({
        "partition": nom,
        "documents": len(docs),
        "docs sans mention": sum(1 for d in docs if not d.mentions),
        "Chemical": par_type["Chemical"],
        "Disease": par_type["Disease"],
        "total mentions": len(mentions),
        "formes distinctes": len({(m.texte.lower(), m.type) for m in mentions}),
        "composites": sum(m.composite for m in mentions),
        "sans id MeSH (-1)": sum(m.identifiants == "-1" for m in mentions),
        "phrases": n_phrases,
        "phrases sans mention": sans_mention,
    })
effectifs = pd.DataFrame(lignes).set_index("partition")
effectifs.to_csv(RESULTATS / "effectifs_corpus.csv")
effectifs.T

partition,train,validation,test
documents,500,500,500
docs sans mention,0,0,0
Chemical,5203,5347,5385
Disease,4182,4244,4424
total mentions,9385,9591,9809
formes distinctes,2391,2239,2355
composites,107,89,57
sans id MeSH (-1),76,59,91
phrases,4580,4592,4821
phrases sans mention,773,787,837


In [6]:
# Longueur des mentions (en caractères) et formes les plus fréquentes de TRAIN
mentions_train = [m for d in corpus["train"] for m in d.mentions]
print(pd.Series([m.fin - m.debut for m in mentions_train]).describe().round(1))
for cat in CATEGORIES:
    top = Counter(m.texte.lower() for m in mentions_train if m.type == cat).most_common(10)
    print(f"\n{cat} — 10 formes les plus fréquentes :", top)

count    9385.0
mean       11.7
std         7.2
min         1.0
25%         8.0
50%        10.0
75%        14.0
max       105.0
dtype: float64

Chemical — 10 formes les plus fréquentes : [('cocaine', 108), ('dopamine', 63), ('nicotine', 58), ('morphine', 53), ('lithium', 49), ('haloperidol', 46), ('warfarin', 42), ('clonidine', 40), ('cisplatin', 38), ('lidocaine', 36)]

Disease — 10 formes les plus fréquentes : [('pain', 87), ('toxicity', 78), ('proteinuria', 66), ('hypotension', 61), ('seizures', 61), ('hypertension', 55), ('seizure', 44), ('myocardial infarction', 32), ('hepatitis', 31), ('bradycardia', 27)]


In [7]:
# Exemples de mentions composites (leur traitement est fixé en section 2.2)
pd.DataFrame([
    {"PMID": d.document_id, "surface": m.texte, "categorie": m.type,
     "sous-mentions (annotation)": " | ".join(m.sous_mentions)}
    for d in corpus["train"] for m in d.mentions if m.composite
][:8])

,PMID,surface,categorie,sous-mentions (annotation)
0,2234245,Ocular and auditory toxicity,Disease,Ocular toxicity | auditory toxicity
1,2234245,visual or auditory toxicity,Disease,visual toxicity | auditory toxicity
2,2505783,hemorrhagic cystitis,Disease,hemorrhagic | cystitis
3,2670794,pulmonary and renal insufficiency,Disease,pulmonary insufficiency | renal insufficiency
4,2670794,pulmonary and renal insufficiency,Disease,pulmonary insufficiency | renal insufficiency
5,2670794,pulmonary and renal insufficiency,Disease,pulmonary insufficiency | renal insufficiency
6,3412544,"cancer of the renal pelvis, ureter or bladder",Disease,cancer of the renal pelvis | cancer of the ure...
7,3412544,cancer of the renal pelvis and bladder,Disease,cancer of the renal pelvis | cancer of the bla...


## Étape 3 — Vérifier dix mentions de TRAIN

On tire 10 mentions au hasard et on vérifie que `texte[debut:fin]` donne bien le mot annoté.

In [8]:
rng = random.Random(SEED)
tirage = rng.sample([(d, m) for d in corpus["train"] for m in d.mentions], 10)
verif = pd.DataFrame([{
    "PMID": d.document_id, "debut": m.debut, "fin": m.fin, "categorie": m.type,
    "surface annotée": m.texte,
    "texte[debut:fin]": d.texte[m.debut:m.fin],
    "identique": d.texte[m.debut:m.fin] == m.texte,
    "contexte": "…" + d.texte[max(0, m.debut - 25):m.debut] + "⟦" + d.texte[m.debut:m.fin]
                + "⟧" + d.texte[m.fin:m.fin + 25] + "…",
} for d, m in tirage])
assert verif["identique"].all()
verif

,PMID,debut,fin,categorie,surface annotée,texte[debut:fin],identique,contexte
0,3676049,580,586,Chemical,oxygen,oxygen,True,"…lus 67% nitrous oxide in ⟦oxygen⟧), during wh..."
1,3827439,268,281,Disease,renal failure,renal failure,True,…s. The abruptness of the ⟦renal failure⟧ and ...
2,17562951,256,271,Chemical,contrast medium,contrast medium,True,…ty of the low-osmolality ⟦contrast medium⟧ io...
3,10704919,1843,1852,Disease,hemolysis,hemolysis,True,…that the protection from ⟦hemolysis⟧ by tocop...
4,7053303,195,205,Chemical,cimetidine,cimetidine,True,…ollected for analysis of ⟦cimetidine⟧ by high...
5,9855119,235,245,Chemical,tacrolimus,tacrolimus,True,…nt patients treated with ⟦tacrolimus⟧ (FK506)...
6,2054792,168,178,Disease,toxicities,toxicities,True,…tivity and normal tissue ⟦toxicities⟧ by whol...
7,9334596,1144,1166,Chemical,testosterone heptylate,testosterone heptylate,True,…ine therapy consisted of ⟦testosterone heptyl...
8,19631624,32,39,Chemical,ecstasy,ecstasy,True,…g and memory deficits in ⟦ecstasy⟧ users and ...
9,18544179,214,247,Disease,postoperative nausea and vomiting,postoperative nausea and vomiting,True,"…thesia with sevoflurane, ⟦postoperative nause..."


**Unité des positions** : caractères, en partant de 0, fin exclue. Exemple (PMID 227508) : « Naloxone » = `[0, 8[`.

Toutes les mentions ont aussi été vérifiées au chargement : aucune n'est décalée.

## Fiche du corpus

Écrite dans `resultats/extraction/fiche_corpus.md`.

In [9]:
e = effectifs
fiche = f"""# Fiche du corpus d'extraction

| Élément | Valeur |
|---|---|
| Nom | BioCreative V CDR (BC5CDR) |
| Version | `CDR.Corpus.v010516` (fichiers PubTator) |
| Source | https://ftp.ncbi.nlm.nih.gov/pub/lu/BC5CDR/ |
| Conditions d'utilisation | Domaine public (« United States Government Work », NCBI/NLM), citation des auteurs demandée |
| Références | Li et al., 2015 (corpus) ; Wei et al., 2015 (tâche CDR) |
| Langue, textes | Anglais ; titres et résumés PubMed |
| Catégories retenues | Chemical, Disease |
| Partitions | Officielles : TRAIN = TrainingSet, VALIDATION = DevelopmentSet, TEST = TestSet (voir 2.2) |
| Unité des positions | Caractères, base 0, `[debut, fin[`, sur `titre + " " + résumé` |
| Mentions composites | {int(e["composites"].sum())} au total, gardées comme une seule mention continue (voir 2.2) |
| Hors socle | Identifiants MeSH (normalisation) et relations CID ignorés |

## Effectifs

| Partition | Documents | Chemical | Disease | Total | Composites | Phrases sans mention |
|---|---|---|---|---|---|---|
"""
for nom, r in e.iterrows():
    fiche += (f"| {nom} | {r['documents']} | {r['Chemical']} | {r['Disease']} | {r['total mentions']} "
              f"| {r['composites']} | {r['phrases sans mention']} / {r['phrases']} |\n")
fiche += """
## Exemple

PMID 227508 : « [Naloxone]<Chemical> reverses the antihypertensive effect of [clonidine]<Chemical>. »
puis dans le résumé « spontaneously [hypertensive]<Disease> rats ».
"""
(RESULTATS / "fiche_corpus.md").write_text(fiche, encoding="utf-8")
print(fiche)

# Fiche du corpus d'extraction

| Élément | Valeur |
|---|---|
| Nom | BioCreative V CDR (BC5CDR) |
| Version | `CDR.Corpus.v010516` (fichiers PubTator) |
| Source | https://ftp.ncbi.nlm.nih.gov/pub/lu/BC5CDR/ |
| Conditions d'utilisation | Domaine public (« United States Government Work », NCBI/NLM), citation des auteurs demandée |
| Références | Li et al., 2015 (corpus) ; Wei et al., 2015 (tâche CDR) |
| Langue, textes | Anglais ; titres et résumés PubMed |
| Catégories retenues | Chemical, Disease |
| Partitions | Officielles : TRAIN = TrainingSet, VALIDATION = DevelopmentSet, TEST = TestSet (voir 2.2) |
| Unité des positions | Caractères, base 0, `[debut, fin[`, sur `titre + " " + résumé` |
| Mentions composites | 253 au total, gardées comme une seule mention continue (voir 2.2) |
| Hors socle | Identifiants MeSH (normalisation) et relations CID ignorés |

## Effectifs

| Partition | Documents | Chemical | Disease | Total | Composites | Phrases sans mention |
|---|---|---|---|---|-